## Анализ данных о Нобелевских лауреатах


### **Этап 1: Подготовка данных**



Собираем данные о нобелевских лауреатах с внешнего API, очищаем и нормализуем их в плоский формат (список словарей). Для этого используем json_dict_processing.py, laureates_configs.py и prizes_configs.py, которые нужны чтобы обрабатывать вложенные JSON-структуры.

In [1]:
import requests
import json
# Импортируем готовые процессоры для людей и организаций
from laureates_configs import process_orgs, process_persons

URL_LAUREATES = 'https://api.nobelprize.org/2.1/laureates?limit=1200'

laureates_response = requests.get(URL_LAUREATES)
laureates = laureates_response.json()
print(f"Данные загружены. Всего записей: {len(laureates.get('laureates', []))}")

/Users/ilzgm/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Данные загружены. Всего записей: 1018


In [2]:
def get_laureate_data(laureate):
    """
    Определяет тип лауреата и применяет соответствующий процессор.
    Принимает на вход словарь с исходными данными лауреата, возвращает обработанный словарь с добавленным полем 'type_' ('person' или 'org')
    или none, если тип не определен.
    """
    if 'knownName' in laureate:
        processed_data = process_persons(laureate)
        processed_data['type_'] = 'person'
        return processed_data
    elif 'orgName' in laureate:
        processed_data = process_orgs(laureate)
        processed_data['type_'] = 'org'
        return processed_data
    else:
        print(f"Внимание: Не удалось определить тип лауреата с ID: {laureate.get('id', 'N/A')}")
        return None


laureates_data = []

for laureate in laureates.get('laureates', []):
    processed = get_laureate_data(laureate)
    if processed:
        laureates_data.append(processed)

print(f"Получили {len(laureates_data)} записей.")

Получили 1018 записей.




### **Этап 2: EDA лауреатов**

## Технический анализ

Проверяем качество данных, смотрим количество пропсков и тд

In [3]:
from typing import Any, Dict, List, Union, Callable, Tuple, Optional

# 1. Сколько всего записей?
total_records = len(laureates_data)
print(f"Общее количество записей: {total_records}")

# 3. Сколько пропусков и какова их доля?
all_keys = set()
for item in laureates_data:
    all_keys.update(item.keys())

missing_counts = {}
for key in sorted(list(all_keys)):
    missing_counts[key] = sum(1 for item in laureates_data if item.get(key) is None)

print("\nКоличество и доля пропусков в каждом поле:")
print({k: missing_counts[k] for k in sorted(missing_counts.keys())})

print("\nДоля пропусков:")
for key, count in sorted(missing_counts.items(), key=lambda x: x[1], reverse=True):
    share = (count / total_records) * 100
    if share > 0:
        print(f"{key}: {count} пропусков, {share:.2f}%")
print('Те которые не вывел = 0%')

# 4. Анализ id
id_list = [item['id'] for item in laureates_data if item['id'] is not None]
id_set = set(id_list)
print(f"\nАнализ ID:")

if len(id_list) != len(id_set):
    print('Есть повторяющиеся ID')
else:
    print('Нет повторяющихся ID')

min_id = min(id_list)
max_id = max(id_list)
print(f"Минимальный ID: {min_id}")
print(f"Максимальный ID: {max_id}")

def find_award_year_by_id(laureates: List[Dict[str, Any]], id_val: int) -> Optional[int]:
    """Ищет год первого приза для лауреата с данным ID."""
    for item in laureates:
        if item.get('id') == id_val and item.get('prizes_relevant'):
            # Сортируем призы по году и берем самый первый
            first_prize = min(item['prizes_relevant'], key=lambda p: p['award_year'])
            return first_prize['award_year']
    return None

min_id_year = find_award_year_by_id(laureates_data, min_id)
max_id_year = find_award_year_by_id(laureates_data, max_id)

print(f"Год первого приза для мин. ID ({min_id}): {min_id_year}")
print(f"Год первого приза для макс. ID ({max_id}): {max_id_year}")

# Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один?
sorted_ids = sorted(id_list)
gaps = []
for i in range(len(sorted_ids) - 1):
    diff = sorted_ids[i+1] - sorted_ids[i]
    if diff > 1:
        gaps.append((sorted_ids[i], sorted_ids[i+1], diff))

if len(gaps) != 0:
    print(f"Есть {len(gaps)} разрывов в ID.")
else:
    print("Разрывов в ID не обнаружено.")

Общее количество записей: 1018

Количество и доля пропусков в каждом поле:
{'birth_year': 28, 'country_birth': 32, 'country_founded': 994, 'country_now': 8, 'founded_year': 991, 'gender': 28, 'id': 0, 'name': 0, 'prizes_relevant': 0, 'type_': 0}

Доля пропусков:
country_founded: 994 пропусков, 97.64%
founded_year: 991 пропусков, 97.35%
country_birth: 32 пропусков, 3.14%
birth_year: 28 пропусков, 2.75%
gender: 28 пропусков, 2.75%
country_now: 8 пропусков, 0.79%
Те которые не вывел = 0%

Анализ ID:
Нет повторяющихся ID
Минимальный ID: 1
Максимальный ID: 1060
Год первого приза для мин. ID (1): 1901
Год первого приза для макс. ID (1060): 2025
Есть 24 разрывов в ID.


# Анализируем датасет

Для анализа используются функции из dataset_helpers.py, а также агрегационные функции из aggregations.py.

# Вспомогательные функции для анализа стран

In [4]:
import dataset_helpers as dh
import aggregations as agg

def filter_by_category(data_list, category):
    """
    Фильтрует лауреатов, которые получали приз в конкретной категории.
    (Лауреат может иметь несколько призов, поэтому проверяем список 'prizes_relevant').
    """
    return [
        laureate for laureate in data_list 
        if any(prize.get('category_en') == category 
               for prize in laureate.get('prizes_relevant', []))
    ]

def output_top_countries(data_list, top_n=5):
    """
    Выводит топ из N стран по количеству лауреатов по country_now.
    """
    top_countries = agg.get_top_n(data_list, 'country_now', n=top_n)
    print(f"Топ-{top_n} по country_now:")
    for rank, (country, count) in enumerate(top_countries, 1):
        print(f"{rank}. {country}: {count} лауреатов")
    return top_countries

# Топ статистика по странам (ответы на вопросы 2.1)

In [5]:
# Топ 5 стран по общему количеству лауреатов
all_laureates = dh.filter_data(laureates_data, {'country_now': lambda x: x is not None})
top_overall = output_top_countries(all_laureates, 5)
print("\n" , "="*50 , "\n")

# Рейтинги по категориям
print('Рейтинги по категориям:')
categories = [
    'Physics', 'Chemistry', 'Physiology or Medicine', 
    'Literature', 'Peace', 'Economic Sciences'
]
for category in categories:
    category_data = filter_by_category(all_laureates, category)
    top_n_for_category = agg.get_top_n(category_data, 'country_now', n=5)
    print(f'\n{category}')
    output_top_countries(category_data, 5)

Топ-5 по country_now:
1. USA: 302 лауреатов
2. United Kingdom: 96 лауреатов
3. Germany: 84 лауреатов
4. France: 65 лауреатов
5. Japan: 31 лауреатов


Рейтинги по категориям:

Physics
Топ-5 по country_now:
1. USA: 71 лауреатов
2. Germany: 27 лауреатов
3. United Kingdom: 25 лауреатов
4. France: 13 лауреатов
5. Japan: 12 лауреатов

Chemistry
Топ-5 по country_now:
1. USA: 59 лауреатов
2. Germany: 26 лауреатов
3. United Kingdom: 26 лауреатов
4. France: 11 лауреатов
5. Japan: 8 лауреатов

Physiology or Medicine
Топ-5 по country_now:
1. USA: 83 лауреатов
2. United Kingdom: 25 лауреатов
3. Germany: 18 лауреатов
4. France: 12 лауреатов
5. Sweden: 8 лауреатов

Literature
Топ-5 по country_now:
1. France: 12 лауреатов
2. USA: 10 лауреатов
3. Sweden: 7 лауреатов
4. Poland: 7 лауреатов
5. Germany: 7 лауреатов

Peace
Топ-5 по country_now:
1. USA: 26 лауреатов
2. France: 12 лауреатов
3. Switzerland: 8 лауреатов
4. United Kingdom: 7 лауреатов
5. Russia: 5 лауреатов

Economic Sciences
Топ-5 по country_n

# Вспомогательные функции для анализа возраста

In [6]:
def get_age_at_prize(laureate):
    """
    Вычисляет возраст лауреата на момент получения первого приза.
    Принимаем на вход бработанный словарь лауреата, а возвращаем возраст в годах или none
    """
    birth_year = laureate.get('birth_year')
    prizes = laureate.get('prizes_relevant')

    if birth_year is None or not prizes:
        return None

    # Ищем минимальный год награждения
    min_award_year = min(prize.get('award_year') for prize in prizes if prize.get('award_year') is not None)
    
    if min_award_year is None:
        return None

    # Возраст = Год первого приза - Год рождения
    age = min_award_year - birth_year
    
    # Возраст не может быть меньше 0
    return age

def filter_by_person(data_list):
    """
    Фильтрует данные, оставляя только лауреатов-людей.
    """
    return dh.filter_data(data_list, {'type_': 'person'})

def add_age_attribute(data_list):
    """
    Добавляет поле 'age_at_first_prize' к каждому лауреату.
    """
    import atribute_manager as am
    return am.add_attribute(data_list, 'age_at_first_prize', get_age_at_prize)

# Возраст при награждении (2.2)

In [7]:
people_laureates = filter_by_person(laureates_data)

people_laureates_with_age = add_age_attribute(people_laureates)

# Фильтруем список, чтобы оставить только тех, для кого возраст рассчитан
valid_age_laureates = dh.filter_data(people_laureates_with_age, 
                                     {'age_at_first_prize': lambda x: x is not None})

# Общий средний возраст
all_ages = dh.get_column_data(valid_age_laureates, 'age_at_first_prize')
overall_mean_age = agg.calculate_mean(all_ages)
overall_median_age = agg.calculate_median(all_ages)

print("Анализ возраста при награждении\n")
print(f"Общий средний возраст получения первого приза: {overall_mean_age:.2f} лет")
print(f"Общая медиана возраста получения первого приза: {overall_median_age:.2f} лет")

# Различия по категориям
categories_for_people = [
    'Physics', 'Chemistry', 'Physiology or Medicine', 
    'Literature', 'Peace', 'Economic Sciences'
]
age_by_category = {}

print("\nСредний и медианный возраст по категориям")

for category in categories_for_people:
    category_data = filter_by_category(valid_age_laureates, category)
    category_ages = dh.get_column_data(category_data, 'age_at_first_prize')
    mean_age = agg.calculate_mean(category_ages)
    median_age = agg.calculate_median(category_ages)
    age_by_category[category] = {'mean': mean_age, 'median': median_age}
    
    if mean_age is not None:
        print(f"  {category.ljust(25)} | Средний: {mean_age:.2f} | Медиана: {median_age:.2f}")
    else:
        print(f"  {category.ljust(25)} | Недостаточно данных для расчета")


# Различаются ли средний и медианный возрасты?
print(f"\nРазница между средним и медианой: {abs(overall_mean_age - overall_median_age)}")


Анализ возраста при награждении

Общий средний возраст получения первого приза: 60.44 лет
Общая медиана возраста получения первого приза: 61.00 лет

Средний и медианный возраст по категориям
  Physics                   | Средний: 57.69 | Медиана: 56.00
  Chemistry                 | Средний: 59.05 | Медиана: 58.00
  Physiology or Medicine    | Средний: 58.92 | Медиана: 58.00
  Literature                | Средний: 64.95 | Медиана: 67.00
  Peace                     | Средний: 60.69 | Медиана: 61.50
  Economic Sciences         | Средний: 67.01 | Медиана: 67.00

Разница между средним и медианой: 0.5585858585858574
